# Runtime Analysis

Reports per-dataset method runtime from `experiment_records.jsonl`. Each record
carries a `timings` block (`dunn_search`, `weighted_voting`, `method_total`);
the class count is the number of per-class Dunn scores. No raw data or
predictions are required.

The table supports the computational-cost discussion: the per-class search time
is bounded (typically one to two seconds), weighted voting is a small share of
the total, and the total method time is governed mainly by the number of
classes -- the same per-class search repeats once per class, so the datasets
with the most classes (Food-101, CIFAR-100, Kuzushiji-49) have the largest
totals.

Absolute times are indicative: they were measured on a shared Google Colab CPU
runtime and vary with hardware, CPU contention, and storage I/O. The stable
content is the shape -- bounded per-class cost and total scaling with the class
count -- not the exact seconds.

In [ ]:
import json
import numpy as np
import pandas as pd

RESULTS_JSONL = "data/results/experiment_records.jsonl"

rows = []
with open(RESULTS_JSONL) as f:
    for line in f:
        rec = json.loads(line)
        t = rec.get("timings", {})
        C = len(rec.get("dunn_scores", {})) or np.nan
        dunn = t.get("dunn_search", np.nan)
        rows.append({
            "dataset": rec["dataset"],
            "C": C,
            "dunn_search": dunn,
            "search_per_class": dunn / C if C and not np.isnan(dunn) else np.nan,
            "weighted_voting": t.get("weighted_voting", np.nan),
            "method_total": t.get("method_total", np.nan),
        })

df = pd.DataFrame(rows)
print(f"{len(df)} records over {df['dataset'].nunique()} datasets")

## Per-dataset runtime table

One row per dataset (means over its 60 runs), sorted by class count `C`, with a
mean bottom row. Columns: number of classes, mean Dunn-search time, mean
per-class search time (`dunn_search / C`), mean weighted-voting time, and mean
method total.

In [ ]:
g = (df.groupby("dataset")
       .agg(C=("C", "first"),
            dunn_search=("dunn_search", "mean"),
            search_per_class=("search_per_class", "mean"),
            weighted_voting=("weighted_voting", "mean"),
            method_total=("method_total", "mean"))
       .reset_index()
       .sort_values("C")
       .reset_index(drop=True))

mean_row = {
    "dataset": "MEAN",
    "C": g["C"].mean(),
    "dunn_search": g["dunn_search"].mean(),
    "search_per_class": g["search_per_class"].mean(),
    "weighted_voting": g["weighted_voting"].mean(),
    "method_total": g["method_total"].mean(),
}
show = pd.concat([g, pd.DataFrame([mean_row])], ignore_index=True)

cols = ["dataset", "C", "dunn_search", "search_per_class", "weighted_voting", "method_total"]
fmt = {"C": "{:.0f}", "dunn_search": "{:.2f}", "search_per_class": "{:.2f}",
       "weighted_voting": "{:.3f}", "method_total": "{:.2f}"}
disp = show[cols].copy()
for c, f in fmt.items():
    disp[c] = disp[c].map(lambda v: f.format(v))
print(disp.to_string(index=False))